# Task 1 — Friend Recommendation (Link Prediction)

**Dataset.** A social network of 7,624 LastFM users from Asian countries, with 27,806
mutual-follower edges ([Last.FM Asia](https://snap.stanford.edu/data/feather-lastfm-social.html)).

**Your job.** Given a user `src` and a list of candidate users, rank the candidates so
that the user's real friend comes out on top.

**Setup.** The 27,806 edges were split 70/15/15 into train/val/test. For every positive
edge we picked one endpoint as the query source and sampled negative candidates from
users that `src` is *not* connected to anywhere in the full graph:

| split | queries | positives : negatives |
|-------|---------|-----------------------|
| train | 19,464  | 1 : 5                 |
| val   | 4,171   | 1 : 20                |
| test  | 4,171   | 1 : 20                |

**Metrics.** Hit@1 and MRR.

**What you implement.** The three methods of `LinkPredictor` in section 3. The parts you write are marked `TODO`. Everything
else is provided, only modify when necessary.

**Note.** Run the cells in order. Please follow the instructions provided in the comments. You are encouraged to use coding agents. 

## 1. Setup

On Google Colab this downloads the data. Running from a local checkout, it finds
the files already there and downloads nothing.

In [ ]:
import json
import urllib.request
from collections import defaultdict
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

GITHUB_REPO = "antman9914/CSE60745-Practice"
BRANCH = "main"

DATA_DIR = Path("data/task1")
REQUIRED = ["split_stats.json", "lp_graph_obs.csv",
            "lp_train.csv", "lp_val.csv", "lp_test.csv"]
RANDOM_SEED = 0

DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in REQUIRED:
    if not (DATA_DIR / name).exists():
        url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{BRANCH}/data/task1/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, DATA_DIR / name)

missing = [n for n in REQUIRED if not (DATA_DIR / n).exists()]
assert not missing, f"could not obtain: {missing}"
assert tuple(map(int, nx.__version__.split(".")[:2])) >= (3, 0), \
    f"this notebook needs networkx >= 3.0, found {nx.__version__}"
print(f"data ready in {DATA_DIR}/ | networkx {nx.__version__}, pandas {pd.__version__}")

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

## 2. Data loading

In [ ]:
def load_task1_data(data_dir=DATA_DIR):
    """Load the two observable graphs and the three query sets.

    Returns
    -------
    graphs : dict[str, networkx.Graph]
        "train"    - all 7,624 nodes and the 70% of edges in the training split.
        "trainval" - the same, plus the 15% of edges in the validation split.
        Validation and test positive edges are absent from "train"; test positive edges
        are absent from "trainval". These are the only two graphs you may look at.
    splits : dict[str, pandas.DataFrame]
        One frame per split with columns query_id, src, dst, label. All rows sharing a
        query_id form one ranking problem with exactly one positive candidate.
    """
    stats = json.loads((data_dir / "split_stats.json").read_text())
    splits = {name: pd.read_csv(data_dir / f"lp_{name}.csv")
              for name in ("train", "val", "test")}

    train_edges = pd.read_csv(data_dir / "lp_graph_obs.csv")[["src", "dst"]]
    val_edges = splits["val"].loc[splits["val"]["label"] == 1, ["src", "dst"]]

    G_train = nx.Graph()
    G_train.add_nodes_from(range(stats["n_nodes"]))
    G_train.add_edges_from(train_edges.itertuples(index=False, name=None))

    G_trainval = G_train.copy()
    G_trainval.add_edges_from(val_edges.itertuples(index=False, name=None))

    return {"train": G_train, "trainval": G_trainval}, splits


graphs, splits = load_task1_data()
G_train, G_trainval = graphs["train"], graphs["trainval"]
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

for name, G in graphs.items():
    n_isolated = sum(1 for _, d in G.degree() if d == 0)
    print(f"G_{name:<9} {G.number_of_edges():>6} edges, {n_isolated:>4} isolated nodes, "
          f"{nx.number_connected_components(G):>4} connected components")
for name, df in splits.items():
    n_q = int(df["label"].sum())
    print(f"{name:>5}: {n_q:>6} queries x {len(df) // n_q:>2} candidates = {len(df):>6} rows")

train_df.head(6)


## 3. Your implementation

Fill in the three methods in LinkPredictor.

`adjacency_lookups(G)` returns adjacency matrix ADJ and degree matrix DEG for input graph:

```python
ADJ, DEG = adjacency_lookups(G)    # ADJ[u] is a set of neighbours, DEG[u] an int
```


In [ ]:
_LOOKUP_CACHE = {}


def adjacency_lookups(G):
    """Return (ADJ, DEG) for a graph: neighbour sets and degrees, cached per graph."""
    if G not in _LOOKUP_CACHE:
        adj = {u: set(G.neighbors(u)) for u in G}
        _LOOKUP_CACHE[G] = (adj, {u: len(adj[u]) for u in G})
    return _LOOKUP_CACHE[G]


In [ ]:
class LinkPredictor:
    """Rank candidate friends for a source user using graph structural properties."""

    # ==================================================================================
    # TODO 1 of 3 - feature engineering.
    # ==================================================================================
    def build_features(self, G, pairs):
        """Turn node pairs into a numeric feature matrix.

        Parameters
        ----------
        G : networkx.Graph
            The graph you may look at for this batch of pairs. 
        pairs : numpy.ndarray of shape (n_pairs, 2)
            Each row is a candidate pair (src, dst) to be described.

        Returns
        -------
        numpy.ndarray of shape (n_pairs, n_features), dtype float
            One row of features per input pair. The number of columns is up to you.

        Notes
        -----
        This method is called for each split, and not always with the same
        graph. Please use `adjacency_lookups(G)` to obtain adjancency and degree matrix.
        """
        raise NotImplementedError("TODO 1: implement build_features")

    # ==================================================================================
    # TODO 2 of 3 - training.
    # ==================================================================================
    def fit(self, X_train, y_train, X_val, y_val):
        """Train a classifier or ranking model on the training pairs.

        Parameters
        ----------
        X_train, X_val : numpy.ndarray
            Feature matrices produced by build_features.
        y_train, y_val : numpy.ndarray of shape (n_pairs,)
            Labels: 1 for a real edge, 0 for a sampled non-edge.

        Notes
        -----
        Store the fitted estimator on self (for example self.model) so that score() can
        use it.

        `evaluate_ranking` and `val_df` are already defined, so you
        can select models using code below:

            metrics, _ = evaluate_ranking(val_df, clf.predict_proba(X_val)[:, 1])

        """
        raise NotImplementedError("TODO 2: implement fit")

    # ==================================================================================
    # TODO 3 of 3 - inference.
    # ==================================================================================
    def score(self, X):
        """Score candidate pairs.

        Parameters
        ----------
        X : numpy.ndarray
            A feature matrix produced by build_features.

        Returns
        -------
        numpy.ndarray of shape (n_pairs,)
            A HIGHER score must mean "more likely to be a real edge". The evaluation
            sorts the candidates of each query by this value in descending order.

        Notes
        -----
        Return a continuous score, not a hard 0/1 prediction.
        """
        raise NotImplementedError("TODO 3: implement score")

## 4. Evaluation Toolset

In [ ]:
def evaluate_ranking(df, scores, seed=RANDOM_SEED):
    """Compute Hit@1 and MRR within each query.

    Candidates are shuffled before sorting so that ties are broken uniformly at random.
    This matters: a model that gives every candidate the same score should land on the
    random baseline, not on whatever order the file happened to store them in.
    """
    rng = np.random.default_rng(seed)
    d = df[["query_id", "label"]].copy()
    d["score"] = scores
    d["tiebreak"] = rng.random(len(d))

    d = d.sort_values(["query_id", "score", "tiebreak"], ascending=[True, False, True])
    d["rank"] = d.groupby("query_id").cumcount() + 1

    pos = d.loc[d["label"] == 1, ["query_id", "rank"]]
    metrics = {
        "n_queries": len(pos),
        "hit@1": float((pos["rank"] == 1).mean()),
        "MRR": float((1.0 / pos["rank"]).mean()),
    }
    return metrics, pos.set_index("query_id")["rank"]


def breakdown_by_source_degree(df, ranks, degrees, bins=(0, 1, 2, 4, 8, 16, np.inf)):
    """Split Hit@1 and MRR by how many friends the source user has in the graph.

    `degrees` is a node -> degree mapping for the graph that split was scored on.
    """
    pos = df[df["label"] == 1].set_index("query_id")
    d = pd.DataFrame({"rank": ranks})
    d["src_degree"] = pos.loc[d.index, "src"].map(degrees).to_numpy()
    d["bucket"] = pd.cut(d["src_degree"], bins=list(bins), right=False)

    return d.groupby("bucket", observed=True).agg(
        n_queries=("rank", "size"),
        hit_at_1=("rank", lambda r: (r == 1).mean()),
        MRR=("rank", lambda r: (1.0 / r).mean()),
    ).round(4)


def compare_breakdowns(before, after):
    """Put two breakdowns side by side and show what moved."""
    return pd.DataFrame({
        "n_queries": before["n_queries"],
        "hit@1_before": before["hit_at_1"],
        "hit@1_after": after["hit_at_1"],
        "delta": (after["hit_at_1"] - before["hit_at_1"]).round(4),
    })


## 5. Running the pipeline

In [ ]:
# Which graph each split is described on. Training and validation queries see the
# training edges only; test queries additionally see the validation edges.
FEATURE_GRAPH = {"train": "train", "val": "train", "test": "trainval"}


def run_pipeline(model, graphs, splits):
    """Build features for every split on its own graph, fit on train, score val and test."""
    pairs = {name: df[["src", "dst"]].to_numpy() for name, df in splits.items()}
    labels = {name: df["label"].to_numpy() for name, df in splits.items()}

    features = {}
    for name in ("train", "val", "test"):
        G = graphs[FEATURE_GRAPH[name]]
        features[name] = np.asarray(model.build_features(G, pairs[name]), dtype=float)
        assert features[name].shape[0] == len(pairs[name]), (
            f"build_features returned {features[name].shape[0]} rows "
            f"for {len(pairs[name])} pairs"
        )
        print(f"{name:>5}: feature matrix {features[name].shape}  "
              f"(graph: G_{FEATURE_GRAPH[name]}, {G.number_of_edges()} edges)")

    n_feat = {v.shape[1] for v in features.values()}
    assert len(n_feat) == 1, f"inconsistent feature count across splits: {n_feat}"

    model.fit(features["train"], labels["train"], features["val"], labels["val"])

    scores = {name: np.asarray(model.score(features[name]), dtype=float).ravel()
              for name in ("val", "test")}
    for name, s in scores.items():
        assert s.shape == (len(pairs[name]),), f"score returned shape {s.shape} for {name}"
    return scores


model = LinkPredictor()
scores = run_pipeline(model, graphs, splits)

## 6. Final test evaluation

Run this **at the end**.

In [ ]:
val_metrics, val_ranks = evaluate_ranking(val_df, scores["val"])
print("validation:", val_metrics)
test_metrics, test_ranks = evaluate_ranking(test_df, scores["test"])
print("validation:", val_metrics)
print("test:      ", test_metrics)

## 7. Cold start: users with few friends

Your overall Hit@1 is an average over very different users. The table below splits the
validation queries by how many friends the source user has in `G_train`. Read it before
you decide your model is finished.

**Improve the low-degree buckets.** Go back to `build_features` in section 3, add whatever
you think will help a user with one, two or three friends, re-run section 5, and come
back here. `compare_breakdowns` shows what actually moved. This exercise is judged on the
validation split — leave section 6 alone while you iterate. A change that lifts the overall
number by riding the 16+ bucket has not solved anything — those users were already being
served well.

Worth thinking about:

- A user with a single friend has no common neighbour with almost any candidate. But that
  one friend is not nothing. What does it let you say about who the user might know?
- When the user's own neighbourhood tells you nothing, what still distinguishes one
  candidate from another? Is that difference something you would want a recommender to
  act on?
- If a feature helps the 16+ bucket and does nothing below it, is it worth keeping?

**Explain the bottom row.** Some source users are isolated in `G_train`: no edges at all.
Look at what your model scores them at, and work out whether *any* feature you could write
would move it. If you conclude it cannot, say precisely why, and say what kind of
information would be needed instead. That answer is worth more than a number.


In [ ]:
degrees_train = dict(G_train.degree())
_, val_ranks = evaluate_ranking(val_df, scores["val"])

baseline_breakdown = breakdown_by_source_degree(val_df, val_ranks, degrees_train)
baseline_breakdown


In [ ]:
# After changing build_features and re-running section 5, run this to see what your
# change did to each kind of user.
_, val_ranks = evaluate_ranking(val_df, scores["val"])
current_breakdown = breakdown_by_source_degree(val_df, val_ranks, degrees_train)

compare_breakdowns(baseline_breakdown, current_breakdown)
